# Day 2 - Toolbox & Evaluation

> Two lab blocks, each running after its deck. Work down in order.
> Cells marked `TODO` are yours to fill in; a `checks.check_ex_*`
> call tells you whether it worked. Stretch sections are optional.

## Setup

Same `coursekit` imports, plus `statsforecast` for the models and
`utilsforecast` for the metrics.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import (MSTL, HistoricAverage, Naive,
                                  RandomWalkWithDrift, SeasonalNaive)
from statsforecast.utils import ConformalIntervals
from statsmodels.stats.diagnostic import acorr_ljungbox
from utilsforecast.losses import mae, mape, mase, rmse, rmsse, scaled_crps

from coursekit import checks
from coursekit import datasets as D
from coursekit import leaderboard as lb
from coursekit import plotting as P

P.use_course_style()

spine = D.spine()
H = 24
train, test = D.train_test(spine, h=H)

# Every forecast today is asked for the same ladder of intervals. 80 is the one
# we read coverage off; the rest are there so Exercise 2.5 can score the whole
# forecast DISTRIBUTION and not just one band.
LEVELS = [20, 40, 60, 80, 95]

print(f"train: {len(train)} months to {train['ds'].max().date()}")
print(f"test : {len(test)} months from {test['ds'].min().date()}")

---
# Lab C - The toolbox

Runs after the third deck. Exercises 2.1 to 2.3, 46 minutes.

---
# Exercise 2.1 - The benchmark floor

*15 minutes.*

Fit all four benchmarks and look at them. Everything for the rest of the course
is measured against these.

In [ ]:
MODELS = ["HistoricAverage", "Naive", "SeasonalNaive", "RWD"]
LABELS = {"HistoricAverage": "Mean", "Naive": "Naive",
          "SeasonalNaive": "Seasonal naive", "RWD": "Drift"}

# TODO: build a StatsForecast object with all four benchmarks and forecast H
#       months ahead from `train`, at every level in LEVELS, keeping fitted values.
BENCHMARKS = ...
sf = ...
fc = ...

checks.check_ex_2_1(fc, MODELS)
fc.head()

In [ ]:
# TODO: plot the last 6 years of training data, the four forecasts, and the
#       held-out actuals on one chart.

**Question.** Two of these are obviously wrong before you compute a single
metric. Which, and why?

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* The **mean** method forecasts a flat line at roughly 160 for a series
currently sitting near 370 - it averages over 37 years of growth, so it is
hopeless on any trending series. The **naive** method forecasts a flat line at
the last value, which throws away the seasonality we spent all of Day 1
establishing. **Drift** at least captures the trend but still ignores season.
Only the **seasonal naive** reproduces the annual shape.

### A fifth model, out of Day 1

You already know how to take this series apart: STL gives you trend, season and
remainder. Ch 5.7 turns that into a *forecasting* method. Strip the season off,
forecast the seasonally adjusted series with something that handles trend -
drift, say - then add last year's seasonal shape back on top.

`MSTL` is that recipe in one object, and nothing in it is new to you.
`RandomWalkWithDrift` is a benchmark you fit ten minutes ago; the seasonal part
is a seasonal naive on the seasonal component.

In [ ]:
# TODO: add the STL route as a fifth model - MSTL, season_length=12, with
#       RandomWalkWithDrift as its trend forecaster - and refit all five.
sf = ...
fc = ...

MODELS = ["HistoricAverage", "Naive", "SeasonalNaive", "RWD", "MSTL"]
LABELS["MSTL"] = "STL + drift"
print(f"{len(MODELS)} models: {', '.join(LABELS[m] for m in MODELS)}")

In [ ]:
# TODO: plot the STL route against the seasonal naive - the model it has to beat
#       - over the holdout.

**Question.** Which of those two tracks the holdout better, and what is the STL
route doing that the seasonal naive cannot? Write your answer down now - you
will be asked to revisit it in Exercise 2.5.

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* The STL route is clearly closer over these 24 months. The seasonal
naive repeats last year's level exactly, so on a series that grows about 6% a
year it starts the horizon low and stays low - the gap is a *bias*, visible as
the forecast sitting under the actuals almost everywhere. The STL route
separates that trend out and lets drift carry it forward, so it keeps the
seasonal shape *and* the growth.

Hold that conclusion loosely. It is one window.

### Stretch - forecasting on a transformed scale

The spine is multiplicative. Forecast the Box-Cox transformed series, then
back-transform. Note that the naive back-transform gives you the **median**, not
the mean.

In [ ]:
# Stretch - your code here.

---
# Exercise 2.2 - Are the residuals white noise?

*10 minutes.*

If a model's residuals still carry structure, the model has not finished.

In [ ]:
fv = sf.forecast_fitted_values()

# TODO: compute the seasonal naive's residuals, plot the three-panel
#       diagnostic, and run a Ljung-Box test at lag 24.
resid = ...
lb_pvalue = ...

print(f"mean residual : {pd.Series(resid).mean():.3f}")
print(f"Ljung-Box p   : {lb_pvalue:.3e}")

checks.check_ex_2_2(resid, lb_pvalue)

In [ ]:
# TODO: do the same for the drift method. Which of the four properties
#       (uncorrelated / zero mean / constant variance / normal) does each satisfy?

**Write your verdict.** For each method, which of the four residual properties
hold, and what does that imply?

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* Neither is close to white noise.

- **Uncorrelated:** fails badly for both - Ljung-Box p is effectively zero and
  the residual ACF has large spikes. There is a great deal of signal left.
- **Zero mean:** the seasonal naive's mean residual is clearly positive, because
  the series trends upward and last year's value is systematically too low. That
  is a *bias*: the forecast will be low every time.
- **Constant variance:** fails - the late-period standard deviation is roughly
  double the early one, because the series itself grew about sixfold. This is
  exactly what the Box-Cox transform in 1.4 addresses.
- **Normal:** roughly, but with heavy tails.

Implication: the benchmark floor is a floor, not a model. The failures are
informative - the bias says "add a trend", the seasonal spikes say "the seasonal
shape has changed", the variance says "transform first".

---
# Exercise 2.3 - Intervals, and how much to believe them

*21 minutes.*

Three ways to draw an interval around the same point forecast, each spending a
different assumption: **Gaussian** (part a), **bootstrap** (part b) and
**conformal** (part c).

*This is the long exercise of the day.* If the room is short on time, **part c
is the one to come back to later** - nothing in 2.4 or 2.5 depends on it.

## Part a - the Gaussian interval

In [ ]:
# TODO: plot the seasonal naive's forecast with 80% and 95% intervals against
#       the held-out actuals. (P.fan_chart wants columns mean / lo-80 / hi-80 / ...)

In [ ]:
# TODO: the 80% interval width at each horizon. Then overlay the two candidate
#       formulas: sqrt(h) (the NAIVE method's) and sqrt(k+1) with
#       k = (h-1) // 12 (the SEASONAL naive's). Only one of them fits.
width = ...

h = np.arange(1, len(width) + 1)
k = (h - 1) // 12
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(h, width, color=P.BLUE, lw=4.5, alpha=0.55, label="actual width")
ax.plot(h, width.iloc[0] * np.sqrt(k + 1), color=P.GREEN, lw=2, dashes=(6, 4),
        label="width_1 * sqrt(k+1)")
ax.plot(h, width.iloc[0] * np.sqrt(h), color=P.ORANGE, ls=":", lw=1.4,
        label="width_1 * sqrt(h)")
ax.set(xlabel="horizon h", ylabel="80% interval width", title="Widening with h")
ax.legend(frameon=False)
plt.show()

In [ ]:
merged = test.merge(fc, on=["unique_id", "ds"])

# TODO: what fraction of the held-out actuals fall inside the 80% interval?
#       And what is the standard error of that estimate?
coverage = ...
se = ...

print(f"nominal 80%,  measured {coverage:.1%}  +/- {1.96 * se:.1%} (95% CI)")
checks.check_ex_2_3(width, coverage, se)

**Question.** Your measured coverage came with an error bar roughly 16 points
wide. What would you have to change to measure coverage properly - and is that
what exercise 2.5 does?

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* You need more scored points, and they must come from *different
origins* rather than from extending one test window (extending it just forecasts
further ahead, where the model is worse). Rolling-origin cross-validation gives
exactly that: 8 folds x 12 months = 96 scored points instead of 24, cutting the
standard error in half. That is exercise 2.5.

Note it does not fix the *other* problem - the interval formula ignores model
uncertainty - so even a well-measured coverage tends to come in under nominal.

---
## Part b - the same interval, without the normality assumption

The Gaussian interval spends three assumptions: uncorrelated residuals, constant
variance, and **normality**. The residual bootstrap buys the third one back. It
resamples the errors you actually saw:

$$y^*_{T+i} = y^*_{T+i-m} + e^*_{T+i}$$

where $e^*$ is drawn at random from the pool of past residuals. Run that
recursion a few thousand times and you have a few thousand possible futures; the
interval is a percentile taken down each column.

> **The assumption you just made.** The simple residual bootstrap assumes the
> residuals come from one common distribution $\hat{F}$ whose distributional
> characteristics **do not change over time** - i.i.d. draws from the pool of
> past errors. Hold on to that; part d comes back to it.

In [ ]:
# TODO: simulate 5000 possible futures for the seasonal naive.
#       P.bootstrap_paths(y, resid, h, season_length=..., n_paths=..., seed=...)
#       returns an (n_paths, h) array.
resid_sn = ...
boot_paths = ...

fig, ax = plt.subplots(figsize=(10, 4))
P.sim_paths_plot(train, fc["ds"], boot_paths, ax=ax, n_show=8, history_tail=60,
                 actual=test, title="Eight of the 5000 simulated futures")
plt.show()

In [ ]:
# TODO: collapse the paths into an interval (P.paths_to_fan) and compare it with
#       the Gaussian one at BOTH levels. Watch what happens between 80% and 95%.
boot_fan = ...

for lvl in (80, 95):
    g = float((fc[f"SeasonalNaive-hi-{lvl}"] - fc[f"SeasonalNaive-lo-{lvl}"]).mean())
    b = float((boot_fan[f"hi-{lvl}"] - boot_fan[f"lo-{lvl}"]).mean())
    print(f"mean {lvl}% width   gaussian {g:5.1f}   bootstrap {b:5.1f}")

checks.check_ex_2_3b(boot_paths, boot_fan, fc)

---
## Part c - $e_{t+h|t}$, and conformal prediction

Conformal prediction throws away the distribution entirely and calibrates on
**$h$-step-ahead forecast errors**:

$$e_{t+h|t} = y_{t+h} - \hat{y}_{t+h|t}$$

- $t$ is when the forecast was **made** (the forecast origin)
- $h$ is the **horizon**, $t+h$ the time being predicted
- $y_{t+h}$ is what happened; $\hat{y}_{t+h|t}$ is the forecast made at $t$ for $t+h$

*Concrete:* you hold $y_1 \dots y_{10}$ and want a 3-step-ahead interval. Stand
at $t = 5$, forecast $\hat{y}_{8|5}$, then look up $y_8$ and record
$e_{8|5} = y_8 - \hat{y}_{8|5}$. Slide the origin to $t = 6, 7, \dots$ and
repeat. At $h = 1$ these are exactly the residuals from exercise 2.2; for
$h > 1$ they are a wider set that has to be **collected**, not fitted.

Build that collection yourself before letting `statsforecast` do it.

In [ ]:
# TODO: roll the origin over the training data and keep only the h = 12 errors.
#       `sf.cross_validation` gives you `cutoff` (the origin t) and `ds` (t+h);
#       the 12-step error is the row where ds is 12 months after cutoff.
cv12 = ...
e12 = ...          # the h = 12 errors themselves, as a Series or array

print("h = 12 errors:", np.round(np.asarray(e12), 1))
print(f"Q_0.80(|e|)  = {np.quantile(np.abs(e12), 0.80):.1f}"
      "   <- half-width of an 80% conformal interval at h = 12")

In [ ]:
# TODO: let statsforecast build the same thing for every horizon at once, with
#       ConformalIntervals(n_windows=8, h=H), then put all three methods in one
#       table: method / width_80 / coverage_80 on the holdout.
fc_conf = ...
cmp = ...

checks.check_ex_2_3c(cmp)
cmp.round(3)

**Question.** Each method spends a different assumption. Name the assumption
each one makes, and say which of them **this series** breaks.

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.*

| Method | Assumes | True here? |
|---|---|---|
| Gaussian | residuals uncorrelated, constant variance, **normal** | no, no, no |
| Bootstrap | residuals uncorrelated, **i.i.d. from $\hat{F}$** | no - the residual SD wanders between 4 and 35 |
| Conformal | past $h$-step errors **exchangeable** with future ones | closest, but a series whose error spread keeps growing is drifting, not exchangeable |

Exchangeability is the weakest of the three: it only asks that the order of the
past errors carries no information, not that they are independent or that they
follow any named distribution. That is why conformal survives this series best.

None of the three is *satisfied* here. The point is not to find a method with no
assumptions - there isn't one - but to know which assumption you are spending
and whether the data supports it.

### Stretch - the assumption is a knob

`P.bootstrap_paths(..., resid_tail=N)` draws only from the last `N` residuals.
If the residual distribution really were constant over time, that would just
throw information away. Sweep `N` and see.

In [ ]:
# Stretch - your code here.

---
# Lab D - Score them honestly

Runs after the fourth deck. Exercises 2.4 and 2.5, 29 minutes.

---
# Exercise 2.4 - Scoring, and the metric that lies

*10 minutes.*

In [ ]:
# TODO: build a table of MAE, RMSE, MAPE, MASE and RMSSE for all five models
#       on the holdout. MASE and RMSSE need seasonality=12 and train_df=train.
scores = ...

checks.check_ex_2_4(scores)
scores.round(3)

Now build the case where MAPE misleads. Construct a near-zero series and two
forecasts: one that is a little too **low**, one that is much too **high**.

In [ ]:
# The same simulated series the slide used: Poisson counts that sit near zero.
low = D.low_volume_demand(n=48, seed=3)
print(low["y"].describe().round(2).to_string())

# TODO: forecast A is always 2.0 units too HIGH; forecast B is always 0.15 too LOW.
#       Compute MAE and MAPE for each. Which does MAE prefer? Which does MAPE prefer?

**Rank the four benchmarks and defend the ranking.** Which metric did you use,
and why not the others?

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* STL + drift > Seasonal naive > Naive > Drift > Mean, on MASE.

MASE, because it is scale-free (so this ranking can be compared against other
series later), it is defined even when the series touches zero, and the
benchmark is built into it - the seasonal naive's MASE of 1.11 immediately tells
you it is still slightly worse than a one-step seasonal naive, while the STL
route's 0.70 says it clears that bar comfortably.

Not MAE or RMSE: correct here, but their units are millions of dollars, so they
cannot be pooled across series. Not MAPE: this series never approaches zero so
it happens to behave, but selecting on MAPE builds a habit that breaks the first
time you meet slow-moving demand.

Note what you have just done: picked a winner off **one** 24-month window. That
is the exact move Exercise 2.5 is about to take apart.

---
# Exercise 2.5 - The harness

*19 minutes.*

This is the exercise the rest of the course rests on. You are building the
evaluation harness that every Day 3 model gets plugged into.

In [ ]:
# TODO: rolling-origin cross-validation over the WHOLE spine:
#       8 origins, 12 months forecast each, every level in LEVELS.
cv = ...

print(f"folds : {cv['cutoff'].nunique()}")
print(f"scored points : {len(cv)}")
cv.head()

Coverage answers one question - *is the 80% band honest?* - and it is blind to
everything else. An interval of plus-or-minus infinity has perfect coverage and
is worth nothing, and two models that both cover 80% can have wildly different
widths. To *rank* forecast distributions you need a proper score.

`scaled_crps` is that score: it averages the quantile (pinball) loss over the
whole ladder of `LEVELS`, so being too wide, too narrow, or centred in the wrong
place all cost you, in one scale-free number. Lower is better.

In [ ]:
# Which quantile each of those interval columns actually is, low to high.
QUANTILES = np.array([0.025, 0.10, 0.20, 0.30, 0.40, 0.60, 0.70, 0.80, 0.90, 0.975])
QCOLS = ["lo-95", "lo-80", "lo-60", "lo-40", "lo-20",
         "hi-20", "hi-40", "hi-60", "hi-80", "hi-95"]


def qcols(model):
    """The ten interval columns of one model, in QUANTILES order."""
    return [f"{model}-{c}" for c in QCOLS]


print(qcols("SeasonalNaive"))

In [ ]:
# TODO: for each model compute, ACROSS FOLDS:
#         - mean MASE   (score each fold against its own training data)
#         - mean RMSSE
#         - scaled CRPS (scaled_crps, pooled over folds, using qcols and QUANTILES)
#         - empirical 80% coverage
#       Return a tidy frame with columns: model / mase / rmsse / crps / coverage_80
summary = ...

checks.check_ex_2_5(cv, summary)
summary.round(3)

**Go back and read your answer from Exercise 2.1.** On the single 24-month
window the STL route beat the seasonal naive on MASE, 0.70 to 1.11. What does
the table above say, and which of the two numbers would you put in front of a
stakeholder?

**And go back to Exercise 2.3.** You measured the seasonal naive's 80% coverage
on a single 24-month window there. Read that number off your own output, put it
next to the `coverage_80` you just computed over 96 points, and account for the
gap.

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* Across eight origins the ordering **flips**: the seasonal naive
averages about 1.18 and the STL route about 1.22, and the seasonal naive wins on
scaled CRPS too. The single window was not a lie - the STL route really was
better over those particular 24 months - it was just one draw from a
distribution wide enough to contain both answers.

The number to report is the eight-fold one, with its spread. The single-window
0.70 is exactly the kind of result that gets a model promoted into production on
the strength of a lucky year.

*Coverage.* On the single window you measured about **96%** (23 of 24 points
inside). Over 96 points it is **77%**. Nothing about the interval changed
between those two numbers - the same model, the same formula. What changed is
how many points the rate was measured on. With n = 24 the standard error of a
coverage estimate is about 8 points, so a genuine 77% band can easily read 96%
on one window, which is exactly what happened. A rate needs a denominator big
enough to be a rate.

Note also that the STL route earns its worse CRPS with *narrower* intervals
(about 40 units wide against the seasonal naive's 49) and worse coverage
(about 61% against 77%). Narrow is not the same as good: CRPS charges you for
the misses that narrowness buys, which is precisely what coverage on its own
cannot tell you.

Write the results to the leaderboard. **This file is the course's running
scoreboard** - Day 3 appends to the same table.

In [ ]:
lb.reset()   # start clean; re-running this cell is safe

for _, row in summary.iterrows():
    lb.record(
        row["model"], day=2,
        mase=float(row["mase"]), rmsse=float(row["rmsse"]),
        crps=float(row["crps"]), coverage_80=float(row["coverage_80"]),
        mase_min=float(row["mase_min"]), mase_max=float(row["mase_max"]),
        notes="Day 2 baseline, 8-fold rolling origin",
    )

table = lb.show()
checks.check_leaderboard(table)
table.round(3)

### Stretch - how much does one window matter?

Score each fold separately, and put your single-window answer from Exercise 2.4
next to the eight-fold one. `P.single_vs_cv_plot` is the helper the slide used.

In [ ]:
# Stretch - your code here.

### Stretch - price the leakage yourself

You have `per_fold`. Score two policies on it. The honest one picks the seasonal
naive once, in advance, and lives with it in every fold. The leaky one picks
whichever model happened to win *that* fold, which is a choice nobody could have
made before seeing the answer.

In [ ]:
# Stretch - your code here. What is the gap worth, as a percentage?

Adding a model to this harness is meant to be a few lines, and it is: build the
`StatsForecast` object, call `cross_validation` with the same `h`, `step_size`,
`n_windows` and `LEVELS`, score it the way the cell above scores the others, and
call `lb.record(...)`. Nothing else in the notebook changes.

You just wrote that scoring loop by hand, which is the point of the exercise.
The same thing lives packaged in `coursekit.scoring`, so Day 3 does not have to
rebuild it:

```python
from coursekit import scoring, leaderboard as lb

cv = sf.cross_validation(df=spine, h=12, step_size=12, n_windows=8,
                         level=scoring.LEVELS)
lb.record("AutoETS", day=3, **scoring.score_cv(cv, "AutoETS", spine))
```

`scoring.QCOLS` and `scoring.QUANTILES` live there too, and they are derived
from `LEVELS` rather than typed out. That matters more than it looks: get the
two out of step and `scaled_crps` returns a number that is wrong and still
positive, which nothing downstream would catch.

One wrinkle worth knowing before Day 3. Not every model produces prediction
intervals on its own - `WindowAverage` raises *"You must pass
`prediction_intervals` to compute them"* if you ask it for a level. The fix is
the argument from Exercise 2.3:

```python
WindowAverage(window_size=12,
              prediction_intervals=ConformalIntervals(n_windows=4, h=12))
```

Any model at all can be given a conformal interval, which is why this harness
can score models it has never met.

---
## End of Day 2

You have an evaluation harness: benchmarks, residual diagnostics, intervals with
an honest error bar, scale-free metrics, and rolling-origin cross-validation.

`labs/leaderboard.csv` now holds the benchmark floor. Every model on Day 3 has
to get past it.